## Model Development

In this notebook, we will begin to develop and train two classification models: Logistic Regression and Gradient Boosting. We will start with the global dataset to establish a baseline performance before applying them to the demographic-specific datasets.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# Load the global datasets created in notebook 2
global_train_df = pd.read_csv("../data/processed/global/global_train.csv")
global_cal_df = pd.read_csv("../data/processed/global/global_cal.csv")
global_test_df = pd.read_csv("../data/processed/global/global_test.csv")


# Seperate the targets from the features
X_train = global_train_df.drop(columns=["Diabetes_binary"])         # Dropping the diabetes classification
y_train = global_train_df["Diabetes_binary"]                        # Keeping only the diabetes classification

X_cal = global_cal_df.drop(columns=["Diabetes_binary"])
y_cal = global_cal_df["Diabetes_binary"]

X_test = global_test_df.drop(columns=["Diabetes_binary"])
y_test = global_test_df["Diabetes_binary"]

# Validate the datasets (# of observations, # of features)
print("Training:", X_train.shape)
print("Training_y:", y_train.shape)
print("Calibration:", X_cal.shape)
print("Calibration_y:", y_cal.shape)
print("Testing:", X_test.shape)
print("Testing_y:", y_test.shape)

Training: (152208, 21)
Training_y: (152208,)
Calibration: (50736, 21)
Calibration_y: (50736,)
Testing: (50736, 21)
Testing_y: (50736,)


Now, we move onto preparing the features for Logistic Regression and Gradient Boosting. For logistic regression, we will scale the features for better results. Gradient boosting generally does not need scaling to create a good model.

In [2]:
# Scale the features for the logistic regression model
scaler = StandardScaler()

# fit_transform() learns feature means and std and scales it
X_train_scaled = scaler.fit_transform(X_train)

# Transform() applies the same values to the calibration and test data
# We dont want to fit the scalar on the calibration or test data to avoid leakage
X_cal_scaled = scaler.transform(X_cal)
X_test_scaled = scaler.transform(X_test)


# Gradient Boosting uses the original unscaled features
X_train_gb = X_train
X_cal_gb = X_cal
X_test_gb = X_test

With the features prepared, we move to training the global models and generating predicted probabilities using the calibration dataset.

In [3]:
# Train the logistic regression model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Train the gradient boosting model
grad_boosting = GradientBoostingClassifier(random_state=42)
grad_boosting.fit(X_train_gb, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


Now we repeat the same training process for each of the demographic-specific models. We are going to train the same two types of models seperatly for each group for comparison later.

In [4]:
# Store the demographic datasets
demographic_datasets = {
    "sex": ["female", "male"],
    "age": ["younger_adult", "middle_age", "older"],
    "income": ["lower_income", "middle_income", "higher_income"],
    "education": ["below_high_school", "high_school", "some_college_or_higher"]
}

# Load the training, calibration, and testing datasets
demographic_data = {}
for category, groups in demographic_datasets.items():
    demographic_data[category] = {}

    for group in groups:
        demographic_data[category][group] = {
            "train": pd.read_csv(f"../data/processed/{category}/{group}_train.csv"),
            "cal": pd.read_csv(f"../data/processed/{category}/{group}_cal.csv"),
            "test": pd.read_csv(f"../data/processed/{category}/{group}_test.csv")
        }

# Now accesses can be done through a nested dictionary ex. demographic_data["sex"]["female"]["train"]

# Store the demographic-specific models
demographic_models = {}

for category, groups in demographic_data.items():
    demographic_models[category] = {}

    for group, datasets in groups.items():

        # Separate features and target of the current group
        X_group_train = datasets["train"].drop(columns=["Diabetes_binary"])
        y_group_train = datasets["train"]["Diabetes_binary"]

        # Scale features for logistic regression for the current group
        group_scaler = StandardScaler()
        X_group_train_scaled = group_scaler.fit_transform(X_group_train)

        # Train logistic regression for the current group
        group_log_reg = LogisticRegression(max_iter=1000, random_state=42)
        group_log_reg.fit(X_group_train_scaled, y_group_train)

        # Train gradient boosting for the current group
        group_gradient_boosting = GradientBoostingClassifier(random_state=42)
        group_gradient_boosting.fit(X_group_train, y_group_train)

        # Store ther current models and scaler
        demographic_models[category][group] = {
            "log_reg": group_log_reg,
            "gradient_boosting": group_gradient_boosting,
            "scaler": group_scaler
        }

        print(f"Trained models for {category}: {group}")

Trained models for sex: female
Trained models for sex: male
Trained models for age: younger_adult
Trained models for age: middle_age
Trained models for age: older
Trained models for income: lower_income
Trained models for income: middle_income
Trained models for income: higher_income
Trained models for education: below_high_school
Trained models for education: high_school
Trained models for education: some_college_or_higher


With all of the models trained we start generating our predictions for each of the models.

In [5]:
# Logistic Regression predictions
global_log_reg_cal = log_reg.predict_proba(X_cal_scaled)[:, 1]              # We only care about the predictions that are 1
global_log_reg_test = log_reg.predict_proba(X_test_scaled)[:, 1]

# Gradient Boosting predictions
global_gb_cal = grad_boosting.predict_proba(X_cal_gb)[:, 1]
global_gb_test = grad_boosting.predict_proba(X_test_gb)[:, 1]

# Store demographic-specific predictions
demographic_predictions = {}

for category, groups in demographic_data.items():
    demographic_predictions[category] = {}

    for group, datasets in groups.items():
        # Separate features and target
        X_group_cal = datasets["cal"].drop(columns=["Diabetes_binary"])
        X_group_test = datasets["test"].drop(columns=["Diabetes_binary"])

        # Retrieve trained models and scaler
        models = demographic_models[category][group]
        group_scaler = models["scaler"]

        # Scale features for Logistic Regression
        X_group_cal_scaled = group_scaler.transform(X_group_cal)
        X_group_test_scaled = group_scaler.transform(X_group_test)

        # Generate predictions
        demographic_predictions[category][group] = {
            "log_reg_cal": models["log_reg"].predict_proba(X_group_cal_scaled)[:, 1],
            "log_reg_test": models["log_reg"].predict_proba(X_group_test_scaled)[:, 1],
            "gb_cal": models["gradient_boosting"].predict_proba(X_group_cal)[:, 1],
            "gb_test": models["gradient_boosting"].predict_proba(X_group_test)[:, 1]
        }

        print(f"Generated predictions for {category}: {group}")


Generated predictions for sex: female
Generated predictions for sex: male
Generated predictions for age: younger_adult
Generated predictions for age: middle_age
Generated predictions for age: older
Generated predictions for income: lower_income
Generated predictions for income: middle_income
Generated predictions for income: higher_income
Generated predictions for education: below_high_school
Generated predictions for education: high_school
Generated predictions for education: some_college_or_higher


Now we want to calibrate the predictions using Platt scaling (sigmoid) and isotonic regression. since we already generated predictions, we'll use CalibratedClassifierCV with FrozenEstimator so we can calibrate the existing trained models without retraining them.

In [6]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Store calibrated global models
global_calibrated_models = {}

# Logistic Regression calibration
global_calibrated_models["log_reg_sigmoid"] = CalibratedClassifierCV(
    FrozenEstimator(log_reg),
    method="sigmoid"
)
global_calibrated_models["log_reg_sigmoid"].fit(X_cal_scaled, y_cal)

global_calibrated_models["log_reg_isotonic"] = CalibratedClassifierCV(
    FrozenEstimator(log_reg),
    method="isotonic"
)
global_calibrated_models["log_reg_isotonic"].fit(X_cal_scaled, y_cal)

# Gradient Boosting calibration
global_calibrated_models["gb_sigmoid"] = CalibratedClassifierCV(
    FrozenEstimator(grad_boosting),
    method="sigmoid"
)
global_calibrated_models["gb_sigmoid"].fit(X_cal_gb, y_cal)

global_calibrated_models["gb_isotonic"] = CalibratedClassifierCV(
    FrozenEstimator(grad_boosting),
    method="isotonic"
)
global_calibrated_models["gb_isotonic"].fit(X_cal_gb, y_cal)\

# Store calibrated demographic-specific models
demographic_calibrated_models = {}

for category, groups in demographic_models.items():
    demographic_calibrated_models[category] = {}

    for group, models in groups.items():
        datasets = demographic_data[category][group]

        # Separate calibration features and target
        X_group_cal = datasets["cal"].drop(columns=["Diabetes_binary"])
        y_group_cal = datasets["cal"]["Diabetes_binary"]

        # Scale features for Logistic Regression
        X_group_cal_scaled = models["scaler"].transform(X_group_cal)

        # Store calibrated models
        demographic_calibrated_models[category][group] = {}

        # Logistic Regression: Platt scaling
        demographic_calibrated_models[category][group]["log_reg_sigmoid"] = CalibratedClassifierCV(
            FrozenEstimator(models["log_reg"]),
            method="sigmoid"
        )
        demographic_calibrated_models[category][group]["log_reg_sigmoid"].fit(
            X_group_cal_scaled, y_group_cal
        )

        # Logistic Regression: Isotonic regression
        demographic_calibrated_models[category][group]["log_reg_isotonic"] = CalibratedClassifierCV(
            FrozenEstimator(models["log_reg"]),
            method="isotonic"
        )
        demographic_calibrated_models[category][group]["log_reg_isotonic"].fit(
            X_group_cal_scaled, y_group_cal
        )

        # Gradient Boosting: Platt scaling
        demographic_calibrated_models[category][group]["gb_sigmoid"] = CalibratedClassifierCV(
            FrozenEstimator(models["gradient_boosting"]),
            method="sigmoid"
        )
        demographic_calibrated_models[category][group]["gb_sigmoid"].fit(
            X_group_cal, y_group_cal
        )

        # Gradient Boosting: Isotonic regression
        demographic_calibrated_models[category][group]["gb_isotonic"] = CalibratedClassifierCV(
            FrozenEstimator(models["gradient_boosting"]),
            method="isotonic"
        )
        demographic_calibrated_models[category][group]["gb_isotonic"].fit(
            X_group_cal, y_group_cal
        )

        print(f"Calibrated models for {category}: {group}")


Calibrated models for sex: female
Calibrated models for sex: male
Calibrated models for age: younger_adult
Calibrated models for age: middle_age
Calibrated models for age: older
Calibrated models for income: lower_income
Calibrated models for income: middle_income
Calibrated models for income: higher_income
Calibrated models for education: below_high_school
Calibrated models for education: high_school
Calibrated models for education: some_college_or_higher


Now we generate calibrated probabilities on the test datasets.

In [7]:
# Store calibrated global predictions
global_calibrated_predictions = {}

for name, model in global_calibrated_models.items():
    # Select the appropriate test features
    if name.startswith("log_reg"):
        X = X_test_scaled
    else:
        X = X_test_gb

    # Generate calibrated probabilities
    global_calibrated_predictions[name] = model.predict_proba(X)[:, 1]

# Store calibrated demographic-specific predictions
demographic_calibrated_predictions = {}

for category, groups in demographic_calibrated_models.items():
    demographic_calibrated_predictions[category] = {}

    for group, models in groups.items():
        # Retrieve the test dataset
        X_group_test = demographic_data[category][group]["test"].drop(
            columns=["Diabetes_binary"]
        )

        # Retrieve the scaler
        scaler = demographic_models[category][group]["scaler"]

        # Scale features for Logistic Regression
        X_group_test_scaled = scaler.transform(X_group_test)

        # Store calibrated predictions
        demographic_calibrated_predictions[category][group] = {
            "log_reg_sigmoid": models["log_reg_sigmoid"].predict_proba(
                X_group_test_scaled
            )[:, 1],

            "log_reg_isotonic": models["log_reg_isotonic"].predict_proba(
                X_group_test_scaled
            )[:, 1],

            "gb_sigmoid": models["gb_sigmoid"].predict_proba(
                X_group_test
            )[:, 1],

            "gb_isotonic": models["gb_isotonic"].predict_proba(
                X_group_test
            )[:, 1]
        }

        print(f"Generated calibrated predictions for {category}: {group}")


Generated calibrated predictions for sex: female
Generated calibrated predictions for sex: male
Generated calibrated predictions for age: younger_adult
Generated calibrated predictions for age: middle_age
Generated calibrated predictions for age: older
Generated calibrated predictions for income: lower_income
Generated calibrated predictions for income: middle_income
Generated calibrated predictions for income: higher_income
Generated calibrated predictions for education: below_high_school
Generated calibrated predictions for education: high_school
Generated calibrated predictions for education: some_college_or_higher


Finally, we can save the predictions so that we can evaluate their performance.

In [8]:
from pathlib import Path

# make folder if not made
path_dir = Path("../data/processed/predictions")
path_dir.mkdir(parents=True, exist_ok=True)

# Store global predictions
global_predictions = pd.DataFrame({
    "y_true": y_test.values,

    "log_reg_uncalibrated": global_log_reg_test,
    "log_reg_sigmoid": global_calibrated_predictions["log_reg_sigmoid"],
    "log_reg_isotonic": global_calibrated_predictions["log_reg_isotonic"],

    "gb_uncalibrated": global_gb_test,
    "gb_sigmoid": global_calibrated_predictions["gb_sigmoid"],
    "gb_isotonic": global_calibrated_predictions["gb_isotonic"]
})

global_predictions.to_csv(
    "../data/processed/predictions/global_predictions.csv",
    index=False
)

# Save demographic-specific predictions
for category, groups in demographic_predictions.items():

    for group, predictions in groups.items():

        # Retrieve actual test labels
        y_group_test = demographic_data[category][group]["test"]["Diabetes_binary"]

        # Combine actual labels and predictions
        results = pd.DataFrame({
            "y_true": y_group_test.values,

            "log_reg_uncalibrated": predictions["log_reg_test"],
            "log_reg_sigmoid": demographic_calibrated_predictions[category][group]["log_reg_sigmoid"],
            "log_reg_isotonic": demographic_calibrated_predictions[category][group]["log_reg_isotonic"],

            "gb_uncalibrated": predictions["gb_test"],
            "gb_sigmoid": demographic_calibrated_predictions[category][group]["gb_sigmoid"],
            "gb_isotonic": demographic_calibrated_predictions[category][group]["gb_isotonic"]
        })

        # Save CSV
        results.to_csv(
            f"../data/processed/predictions/{category}_{group}_predictions.csv",
            index=False
        )